In [1]:
from pathlib import Path
img_path = Path("data/things-eeg2/imgs")

all_paths_train = sorted((img_path / "training_images").rglob("*.jpg"))
all_paths_test = sorted((img_path / "test_images").rglob("*.jpg"))

all_paths_test[:3]

[PosixPath('data/things-eeg2/imgs/test_images/00001_aircraft_carrier/aircraft_carrier_06s.jpg'),
 PosixPath('data/things-eeg2/imgs/test_images/00002_antelope/antelope_01b.jpg'),
 PosixPath('data/things-eeg2/imgs/test_images/00003_backscratcher/backscratcher_01b.jpg')]

In [2]:
from brain_image.model.img_encoder import load_image_encoder, BaseImageEncoder

clip_img_encoder = load_image_encoder("clip_vitl14")
clip_img_encoder.to("cuda")


OptimizedModule(
  (_orig_mod): CLIPImageEncoder(
    (model): CLIPVisionModelWithProjection(
      (vision_model): CLIPVisionTransformer(
        (embeddings): CLIPVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
          (position_embedding): Embedding(257, 1024)
        )
        (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (encoder): CLIPEncoder(
          (layers): ModuleList(
            (0-23): 24 x CLIPEncoderLayer(
              (self_attn): CLIPAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
              )
              (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    

In [9]:
import torch
from brain_image.data import batch_load_images
import tqdm

@torch.no_grad()
def embed_images(img_paths: list[Path], batch_size: int, encoder: BaseImageEncoder, use_progressbar: bool = True, device="cuda"):
    N, B = len(img_paths), batch_size
    all_embeds = []

    with tqdm.tqdm(total=N, disable=not use_progressbar, desc="Embedding data...") as pbar:
        for i in range(0, N, B):
            imgs = batch_load_images(img_paths[i:i+B]).to(device)
            embeds = encoder.encode(imgs).detach().cpu()
            all_embeds.append(embeds)
            pbar.update(B)

    return torch.cat(all_embeds, dim=0)

B = 512

LOAD_CACHE_IF_EXISTS = True
_cached_train_imgs_path = Path("tmp/all_imgs_train.pt")
if LOAD_CACHE_IF_EXISTS and _cached_train_imgs_path.exists():
    all_imgs_train = torch.load(_cached_train_imgs_path)
else:
    all_imgs_train = embed_images(all_paths_train, B, clip_img_encoder)
    torch.save(all_imgs_train, _cached_train_imgs_path)

In [10]:
all_imgs_test = embed_images(all_paths_test, B, clip_img_encoder)

Embedding data...: 512it [00:04, 126.39it/s]              


In [58]:
from dataclasses import dataclass

from typing import Any, Literal
from torch import nn
from torch import Tensor

from diffusers.models.embeddings import TimestepEmbedding, Timesteps
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler

from brain_image.configs import get_device


def get_activation(name: str):
    match name:
        case "silu":
            return nn.SiLU
        case "elu":
            return nn.ELU
        case "gelu":
            return nn.GELU
        case "relu":
            return nn.ReLU
        case _:
            raise NotImplementedError(f"No activation function for {name}")


@dataclass
class DiffusionPriorConfig:
    d_input: int = 768
    d_cond: int = 768
    d_time: int = 128
    d_embed: int = 768
    d_hidden_start: int = 1024
    d_hidden_scale: float = 0.5
    depth: int = 5
    act_func: Literal["silu", "elu", "gelu", "relu"] = "silu"
    dropout: float = 0.0


class SimpleDiffusionPrior(nn.Module):
    class EncoderLayer(nn.Module):
        def __init__(
            self,
            input_encoder: nn.Module,
            time_encoder: nn.Module,
            cond_encoder: nn.Module,
        ):
            super().__init__()
            self.input_encoder = input_encoder
            self.time_encoder = time_encoder
            self.cond_encoder = cond_encoder

        def forward(self, x, time_latent: Tensor, cond_latent: Tensor | None = None):
            time_embed = self.time_encoder(time_latent)
            cond_embed = (
                self.cond_encoder(cond_latent) if cond_latent is not None else 0
            )
            return self.input_encoder(x + time_embed + cond_embed)

    def __init__(self, config: DiffusionPriorConfig = DiffusionPriorConfig()):
        super().__init__()
        self._dummy_param = nn.Parameter(torch.empty(0))
        self.config = config
        act_func = get_activation(config.act_func)

        self.time_proj = Timesteps(
            config.d_time, flip_sin_to_cos=True, downscale_freq_shift=0
        )
        self.input_proj = nn.Sequential(
            nn.Linear(config.d_embed, config.d_hidden_start),
            nn.LayerNorm(config.d_hidden_start),
            act_func(),
        )

        encoder_layers = []
        decoder_layers = []
        
        self.hidden_dims = [int(config.d_hidden_start * config.d_hidden_scale ** d) for d in range(config.depth)]
        self.time_encoders = nn.ModuleList([
            TimestepEmbedding(config.d_time, hidden_dim) for hidden_dim in self.hidden_dims
        ])
        self.cond_encoders = nn.ModuleList([
            nn.Linear(config.d_cond, hidden_dim) for hidden_dim in self.hidden_dims
        ])


        for d in range(config.depth-1):
            hidden_dim = self.hidden_dims[d]
            next_hidden_dim = self.hidden_dims[d+1]
            
            encode_layer = nn.Sequential(
                nn.Linear(hidden_dim, next_hidden_dim),
                nn.LayerNorm(next_hidden_dim),
                act_func(),
                nn.Dropout(config.dropout),
            )
            decode_layer = nn.Sequential(
                nn.Linear(next_hidden_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                act_func(),
                nn.Dropout(config.dropout),
            )

            encoder_layers.append(
                self.EncoderLayer(encode_layer, self.time_encoders[d], self.cond_encoders[d])
            )
            decoder_layers.insert(
                0, self.EncoderLayer(decode_layer, self.time_encoders[d+1], self.cond_encoders[d+1])
            )

        self.encoder_layers = nn.ModuleList(encoder_layers)
        self.decoder_layers = nn.ModuleList(decoder_layers)

        self.out_proj = nn.Linear(config.d_hidden_start, config.d_input)

    def forward(
        self,
        x: Tensor,  # <B, D>
        timestep: torch.Tensor,  # <B> or scalar
        conditioning: torch.Tensor | None = None,  # <B, E>
    ) -> Tensor:
        x = self.input_proj(x)
        batch_size = x.size(0)

        if timestep.dim() == 0:
            time_is_scalar = True
            timestep = timestep.view(1)
        elif timestep.size(0) != x.size(0):
            raise ValueError(f"Expected batch size {batch_size}, received timestep with batch size {timestep.size(0)}")
        else:
            time_is_scalar = False

        if conditioning is not None and conditioning.size(0) != batch_size:
            raise ValueError(f"Expected batch size {batch_size}, received conditioning with batch size {conditioning.size(0)}")
            
        t = self.time_proj(timestep)
            
        if time_is_scalar:
            t = t.expand(batch_size, -1)

        activations = []
        for encoder_layer in self.encoder_layers:
            activations.append(x)
            x = encoder_layer(x, t, conditioning)

        for decoder_layer, activation in zip(
            self.decoder_layers, reversed(activations)
        ):
            x = decoder_layer(x, t, conditioning) + activation

        x = self.out_proj(x)

        return x

    @torch.no_grad()
    def generate(
        self,
        scheduler: DDPMScheduler,
        num_steps: int | None = 50,
        guidance_scale: float = 5.0,
        conditioning: Tensor | None = None,
        batch_size: int | None = None,
        scheduler_timesteps: list[int] | None = None,
        scheduler_kwargs: dict[str, Any] = {},
        generator: torch.Generator | None = None,
        use_progress_bar: bool = False
    ) -> Tensor:
        device = self._dummy_param.device

        # Validate inputs
        if conditioning is None:
            if batch_size is None:
                raise ValueError(
                    f"Need to define either 'conditioning' or 'batch_size'"
                )

            latent_dim = self.config.d_embed

        else:
            batch_size = conditioning.size(0)
            latent_dim = conditioning.size(1)

            if latent_dim != self.config.d_embed:
                raise ValueError(
                    f"Expected conditioning with dim {self.config.d_embed}, received dim {latent_dim}"
                )

        if num_steps is None and scheduler_timesteps is None:
            raise ValueError(
                f"Need to define either 'num_steps' or 'scheduler_timesteps'"
            )

        if scheduler_timesteps is not None:
            num_steps = None

        # Prepare latents and timesteps
        scheduler.set_timesteps(
            num_inference_steps=num_steps,
            timesteps=scheduler_timesteps,
            device=device,
            **scheduler_kwargs,
        )
        timesteps = scheduler.timesteps
        num_steps = len(timesteps)

        latent = torch.randn(batch_size, latent_dim, generator=generator, device=device)

        # Reverse diffusion
        for t in tqdm.tqdm(timesteps, desc="Denoising latent", disable=not use_progress_bar):
            if conditioning is None or guidance_scale == 0:
                noise_pred = self.forward(latent, t)
            
            else:   # Classifier Free Guidance
                noise_pred_cond = self.forward(latent, t, conditioning)
                noise_pred_uncond = self.forward(latent, t)
                noise_pred = noise_pred_uncond + (noise_pred_cond - noise_pred_uncond) * guidance_scale

            latent = scheduler.step(noise_pred, int(t), latent, generator=generator).prev_sample

        return latent


prior = SimpleDiffusionPrior()
sched = DDPMScheduler()
prior.generate(sched, batch_size=1024).shape

torch.Size([1024, 768])

In [59]:
import multiprocessing as mp
from torch.utils.data import Dataset, DataLoader
class DiffusionDataset(Dataset):
    def __init__(self, cond_latents: torch.Tensor, target_latents: torch.Tensor):
        super().__init__()

        assert len(cond_latents) == len(target_latents)
        self.cond_latents = cond_latents
        self.target_latents = target_latents

    def __len__(self):
        return len(self.cond_latents)

    def __getitem__(self, idx):
        return {
            "target_latent": self.target_latents[idx],
            "cond_latent": self.cond_latents[idx]
        }

batch_size = 512
train_dataset = DiffusionDataset(all_imgs_train, all_imgs_train)
test_dataset = DiffusionDataset(all_imgs_test, all_imgs_test)
train_loader = DataLoader(train_dataset, batch_size, shuffle=True, num_workers=mp.cpu_count(), pin_memory=True, persistent_workers=True)
test_loader = DataLoader(test_dataset, shuffle=False, num_workers=mp.cpu_count())

In [ ]:
from diffusers.optimization import get_cosine_schedule_with_warmup

def train_prior(
    prior: SimpleDiffusionPrior,
    num_epochs: int,
    lr: float,
    train_loader: DataLoader,
    test_loader: DataLoader,
    scheduler: DDPMScheduler,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    lr_scheduler: torch.optim.lr_scheduler.LRScheduler,
    cond_drop_prob: float = 0.1,
    use_progress_bar: bool = True
): 
    device = prior.device
    num_training_timesteps = scheduler.config.get("num_training_steps")
    assert num_training_timesteps

    with tqdm.tqdm(total=num_epochs) as pbar:
        for epoch in range(num_epochs):
            # Training step
            prior.train()
            train_loss = 0
            for batch in train_loader:
                cond_embed = batch.get("cond_latent")
                target_embed = batch.get("target_latent")
                if torch.rand(1) < cond_drop_prob:
                    cond_embed = None

                if cond_embed is not None:
                    cond_embed.to(device)
                target_embed.to(device)

                batch_size = target_embed.size(0)

                noise = torch.randn_like(target_embed)
                timesteps = torch.randint(0, num_training_timesteps, (batch_size,), device=device)

                

            # Validation step
            prior.eval()
            for batch in test_loader:
                cond_embed = batch.get(cond_embed)
                if cond_embed is not None:
                    cond_embed.to(device)

            pbar.set_description(f"train_loss: {train_loss_value} | val_loss: {val_loss_value}")



num_epochs = 150
lr = 1e-3
cond_drop_prob = 0.1
criterion = nn.MSELoss()
warmup_steps = 500
train_timesteps: int = 1000

prior = SimpleDiffusionPrior()
prior.to("cuda")
optimizer = torch.optim.Adam(prior.parameters())
scheduler = DDPMScheduler(num_train_timesteps=1000)
lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=(len(train_loader) * num_epochs),
)

train_prior(
    prior=prior,
    num_epochs=num_epochs,
    lr=lr,
    train_loader=train_loader,
    test_loader=test_loader,
    scheduler=scheduler,
    optimizer=optimizer,
    criterion=criterion,
    lr_scheduler=lr_scheduler,
    cond_drop_prob=cond_drop_prob,
)